In [2]:
import pandas as pd

def fill_jyn_column(file1_path, file2_path, output_path):
    # Load both Excel files
    df1 = pd.read_excel(file1_path)
    print(df1)
    df2 = pd.read_excel(file2_path)
    print(df2)
# Specify columns to join on, assuming they are named differently in each file
    chassis_column_file1 = 'Chassis Number'  # Column name in file 1
    chassis_column_file2 = 'Chasis'  # Column name in file 2
    jyn_column_file2 = 'J y N'  # Column to retrieve from file 2



    # Merge based on specified columns, adding the 'JyN' column to df1 where chassis numbers match
    merged_df = pd.merge(df1, 
                         df2[[chassis_column_file2, jyn_column_file2]], 
                         left_on=chassis_column_file1, 
                         right_on=chassis_column_file2, 
                         how='left')

    # Drop the extra column that was used for joining from df2
    merged_df = merged_df.drop(columns=[chassis_column_file2 + '_y'], errors='ignore')
    merged_df = merged_df.drop(columns='N y J')
    merged_df = merged_df.drop(columns='Chasis')
    
    # Save the updated DataFrame to a new Excel file
    merged_df.to_excel(output_path, index=False)
    print(merged_df)
    
# Usage example
file1_path = '/home/gestell3/Desktop/TODAS FACTURAS VW/P1 - PDFs/SEAT OCT24_Facturas Ny J.xlsx'  # Replace with your file path
file2_path = '/home/gestell3/Desktop/ExtraccionGeneralTODO.xlsx'  # Replace with your file path
output_path = '/home/gestell3/Desktop/TODAS FACTURAS VW/P1 - PDFs/SEAT OCT24_Facturas Ny J EDITED.xlsx'  # Replace with desired output file path

fill_jyn_column(file1_path, file2_path, output_path)


         Chassis Number  Invoice  N y J
0     VSSBAAKM5RR145635  444397E    NaN
1     VSSCE6KJ3RR217184  448489E    NaN
2     VSSCE6KJ6RR218216  448490E    NaN
3     VSSCC75F6R6584648  464839E    NaN
4     VSSBF6KJ8RR212068  448475E    NaN
...                 ...      ...    ...
2367  VSSDE6KJ9RR213306  444365E    NaN
2368  VSSDG6KJ6RR219249  448494E    NaN
2369  VSSJF6KJ8RR217413  444372E    NaN
2370  VSSDG6KJ8RR214960  448482E    NaN
2371  VSSCF6KJ8RR212916  444374E    NaN

[2372 rows x 3 columns]
        Factura Pais     Fecha    Auto             Chasis J y N      Amount  \
0       059665E  DEU  170823.0  KN25Y3  VSSCA7KN5RW002069     J  483.681,00   
1       059938E  DEU  170823.0  KN25Y3  VSSCA7KN2RW002790     J  488.766,00   
2       060035E  DEU  180823.0  KN25Y3  VSSCA7KN8RW002129     J  483.681,00   
3       060671E  CZE  180823.0  KHP27Q  VSSAC75F9R6509675     J  366.953,00   
4       060934E  DEU  180823.0  KN25Y3  VSSCA7KN7RW002199     J  483.681,00   
...         ...  ... 

In [27]:
import pandas as pd
import re
import itertools
import numpy as np


def Concentrado1Mod(file_path_DWH, file_path_Divisiones):
    # Load Data
    df_data_warehouse = pd.read_csv(file_path_DWH, header=None, encoding='unicode_escape')
    df_divisiones = pd.read_excel(file_path_Divisiones, engine='openpyxl')
    
    print("Data Warehouse head:\n", df_data_warehouse.head())  # Check data loaded correctly
    print("Divisiones head:\n", df_divisiones.head())

    list_str = list(df_data_warehouse[0])
    list_index = list(df_data_warehouse.index)

    # Column Creations
    list_col_A = [i[:2] for i in list_str]
    list_col_B = [i[2:8] for i in list_str]
    list_col_C = [i[8:16].replace(' ', '') for i in list_str]
    list_col_D = [str(i[20:22] + i[18:20] + i[16:18]) for i in list_str]
    list_col_E = [i[22:39] for i in list_str]
    list_col_F = [(i[39:46] + '.' + i[46:49]).replace(' ', '') for i in list_str]

    print("List Column A sample:", list_col_A[:5])  # Checking the first few entries
    print("List Column B sample:", list_col_B[:5])

    # Process Columns J and K
    list_recortado = [re.findall(r'\w+-\d', i) for i in list_str]
    list_recortado_aplanado = list(itertools.chain(*list_recortado))
    list_recortado2 = [re.sub(r'[a-zA-Z]', '', i) for i in list_recortado_aplanado]
    
    list_col_J = [i[:4] for i in list_recortado2]
    list_col_K = [i[4:11] for i in list_recortado2]

    # Extracting Fecha Pedimento (Column L)
    list_col_L = [i[81:89] if len(i) >= 56 else np.nan for i in list_str]
    
    # Division Columns
    list_cve_divisiones = list(df_divisiones['CLAVES'].astype(str))
    list_tip_divisiones = list(df_divisiones['Tipo'])
    list_frac_divisiones = list(df_divisiones['FRACCIÓN'])
    list_pais_divisiones = list(df_divisiones['Pais'])
    list_seguro_divisiones = list(df_divisiones['Seguro (Incrementables)'])
    list_flete_divisiones = list(df_divisiones['Flete (Incrementables)'])
    list_marca_divisiones = list(df_divisiones['MARCA'])

    # Match Divisions and track any unmatched rows
    list_col_G, list_col_H, list_col_I = [], [], []
    list_col_M, list_col_N, list_col_EXTRA1 = [], [], []
    index_encontrado = []

    for i, val_B in enumerate(list_col_B):
        matched = False
        for j, val_div in enumerate(list_cve_divisiones):
            if val_B in val_div:
                list_col_G.append(list_tip_divisiones[j])
                list_col_H.append(list_frac_divisiones[j])
                list_col_I.append(list_pais_divisiones[j])
                index_encontrado.append(list_index[i])
                list_col_M.append(list_flete_divisiones[j])
                list_col_N.append(list_seguro_divisiones[j] / 100)
                list_col_EXTRA1.append(list_marca_divisiones[j])
                matched = True
                break
        if not matched:
            list_col_G.append(np.nan)
            list_col_H.append(np.nan)
            list_col_I.append(np.nan)
            list_col_M.append(np.nan)
            list_col_N.append(np.nan)
            list_col_EXTRA1.append(np.nan)

    # Log the lengths of lists before concatenation to identify any mismatches
    print("Length checks:")
    print("List A:", len(list_col_A), "List B:", len(list_col_B), "List C:", len(list_col_C))
    print("List G:", len(list_col_G), "List H:", len(list_col_H), "List I:", len(list_col_I))

    # Converting lists to DataFrames for concatenation
    df_col_A = pd.DataFrame(list_col_A)
    df_col_B = pd.DataFrame(list_col_B)
    df_col_C = pd.DataFrame(list_col_C)
    df_col_D = pd.DataFrame(list_col_D)
    df_col_E = pd.DataFrame(list_col_E)
    df_col_F = pd.DataFrame(list_col_F)
    df_col_G = pd.DataFrame(list_col_G)
    df_col_H = pd.DataFrame(list_col_H)
    df_col_I = pd.DataFrame(list_col_I)
    df_col_J = pd.DataFrame(list_col_J)
    df_col_K = pd.DataFrame(list_col_K)
    df_col_L = pd.DataFrame(list_col_L)
    df_col_M = pd.DataFrame(list_col_M)
    df_col_N = pd.DataFrame(list_col_N)
    df_col_O = pd.DataFrame([np.nan] * len(list_col_A))  # Placeholder
    df_col_P = pd.DataFrame(list_col_EXTRA1)

    # Concatenate columns
    df_concentrado = pd.concat([df_col_A, df_col_B, df_col_C, df_col_D, df_col_E, df_col_F, 
                                df_col_G, df_col_H, df_col_I, df_col_J, df_col_K, df_col_L,
                                df_col_M, df_col_N, df_col_O, df_col_P], axis=1)
    
    df_concentrado.columns = ['ID', 'AUTO', 'FACT', 'FECFACT', 'CHASIS', 'PRECIO', 'TIPO', 
                              'FRACCION', 'PAIS', 'PATENTE', 'PEDIMENTO', 'FECHA PEDIMENTO', 
                              'FLETES', 'SEGUROS', 'ADUANA', 'MARCA']

    return [df_concentrado, df_data_warehouse]


import pandas as pd
from PyPDF2 import PdfFileReader
import re
from typing import List

def estadistico_v1(Concentrado2_path: str, PDF_sec_economia_path: str, output_file: str):
    #####################
    # LECTURA DE DATOS
    #####################
    try:
        df_pedimentos = pd.read_excel(Concentrado2_path, engine='openpyxl', header=0, index_col=0)
    except Exception as e:
        raise Exception(f"Error al leer el archivo Excel: {e}")
    df_pedimentos['FECHA PEDIMENTO'] = df_pedimentos['FECHA PEDIMENTO'].astype(str)

    # Extracción de texto del PDF
    try:
        with open(PDF_sec_economia_path, 'rb') as archivo_pdf:
            lector_pdf = PdfFileReader(archivo_pdf)
            num_paginas = lector_pdf.getNumPages()
            list_text1 = []
            for pagina_num in range(num_paginas):
                pagina = lector_pdf.getPage(pagina_num)
                text = pagina.extractText()
                list_text1.append(text)
            cadena1 = " ".join(list_text1)
    except Exception as e:
        raise Exception(f"Error al leer el archivo PDF: {e}")

    # Procesando texto extraído del PDF
    patron_cupo = r'\bcertificado de cupo con número de autorización \w+/\w+'
    match_cupo = re.findall(patron_cupo, cadena1)
    match_cupo3 = match_cupo[0].split(' ')[-1] if match_cupo else 'N/A'
    
    patron_pieza = r'\w+ Pieza.'
    match_pieza = re.findall(patron_pieza, cadena1)
    match_pieza3 = match_pieza[0][8:].replace(' Pieza.', '') if match_pieza else 'N/A'
    
    # Generación de archivos mensuales
    with pd.ExcelWriter(output_file) as writer:
        for month in range(1, 13):
            month_str = f"{month:02d}"
            print(month_str)
            # Filtrar por mes en FECHA PEDIMENTO
            df_pedimentos_mes = df_pedimentos[df_pedimentos['FECHA PEDIMENTO'].str[4:6] == month_str]
            
            # Filtrando datos por país y marca
            df_pedimentos_India = df_pedimentos_mes[df_pedimentos_mes['PAIS'] == 'IND']
            df_pedimentos_Sudafrica = df_pedimentos_mes[df_pedimentos_mes['PAIS'] == 'ZAF']
            df_pedimentos_NO_India = df_pedimentos_mes[df_pedimentos_mes['PAIS'] != 'IND']
            df_pedimentos_NO_Sudafrica = df_pedimentos_NO_India[df_pedimentos_NO_India['PAIS'] != 'ZAF']
            df_pedimentos_N = df_pedimentos_NO_Sudafrica[df_pedimentos_NO_Sudafrica['J y N'] == 'N']
            df_pedimentos_N_excepcion = df_pedimentos_N[df_pedimentos_N['FRACCION'] != 8703800100]
            
            df_pedimento_Audi = df_pedimentos_N_excepcion[df_pedimentos_N_excepcion['MARCA'] == 'AUDI']
            df_pedimento_Bentley = df_pedimentos_N_excepcion[df_pedimentos_N_excepcion['MARCA'] == 'BENTLEY']
            df_pedimento_Porsche = df_pedimentos_N_excepcion[df_pedimentos_N_excepcion['MARCA'] == 'PORSCHE']
            df_pedimento_Seat = df_pedimentos_N_excepcion[df_pedimentos_N_excepcion['MARCA'] == 'SEAT']
            df_pedimento_Volkswagen = df_pedimentos_N_excepcion[df_pedimentos_N_excepcion['MARCA'] == 'VOLKSWAGEN']
            
            # Cálculos
            suma_descargo = (len(df_pedimentos_India) + len(df_pedimentos_Sudafrica) + 
                             len(df_pedimento_Bentley) + len(df_pedimento_Audi) + 
                             len(df_pedimento_Porsche) + len(df_pedimento_Seat) + 
                             len(df_pedimento_Volkswagen))
            
            Subtotal_desglose_Europa = (len(df_pedimento_Bentley) + len(df_pedimento_Audi) + 
                                        len(df_pedimento_Porsche) + len(df_pedimento_Seat) + 
                                        len(df_pedimento_Volkswagen))
            Diferencia_cupo_desgloses = float(match_pieza3) - suma_descargo
            
            # Formato del estadístico para el mes
            list_col_A = ['', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', 
                          'TOTAL CUPO ' + match_pieza3, match_cupo3]
            list_col_B = ['', f'Periodo {month_str} 2024', '', f'Descargo {month_str} 2024', '', 'DESGLOSE EUROPA:', 
                          'Audi', 'Bentley', 'Porsche', 'Seat', 'Vehiculos Comerciales', 'Subtotal',
                          'Desglose INDIA:', 'India', 'Subtotal', 'Desglose SUDAFRICA', 'Sudafrica', 'Subtotal',
                          'TOTAL', '', 'CUPO', 'SALDO']
            list_col_C = ['', '',  '', suma_descargo, '', '', len(df_pedimento_Audi), len(df_pedimento_Bentley), 
                          len(df_pedimento_Porsche), len(df_pedimento_Seat), len(df_pedimento_Volkswagen),
                          Subtotal_desglose_Europa, '', len(df_pedimentos_India), len(df_pedimentos_India), 
                          '', len(df_pedimentos_Sudafrica), len(df_pedimentos_Sudafrica), suma_descargo, '', 
                          match_pieza3, Diferencia_cupo_desgloses]
            list_col_D = ['', '', '', 'vehiculos', '', '', '', '', '', '', '', 'vehiculos', '', '', '', '', '', '',
                          'vehiculos', '', 'vehiculos', '']
            df_col_completo_pestana = pd.DataFrame([list_col_A, list_col_B, list_col_C, list_col_D]).T
            
            # Guardar en la hoja del mes
            df_col_completo_pestana.to_excel(writer, sheet_name=f'Mes_{month_str}', index=False)
            
    print(f"Estadístico mensual guardado en '{output_file}' para cada mes del año.")

    
    import pandas as pd
import re
from PyPDF2 import PdfFileReader

import pandas as pd
import re
from PyPDF2 import PdfFileReader

def estadistico_v2(Concentrado2_path: str, PDF_sec_economia_path: str, output_file: str,PDF_sec_economia_path2: str):
    #####################
    # LECTURA DE DATOS
    #####################
    try:
        df_pedimentos = pd.read_excel(Concentrado2_path, engine='openpyxl', header=0, index_col=0)
    except Exception as e:
        raise Exception(f"Error al leer el archivo Excel: {e}")
    df_pedimentos['FECHA PEDIMENTO'] = df_pedimentos['FECHA PEDIMENTO'].astype(str)

    # Extracción de texto del PDF 1 
    try:
        with open(PDF_sec_economia_path, 'rb') as archivo_pdf:
            lector_pdf = PdfFileReader(archivo_pdf)
            num_paginas = lector_pdf.getNumPages()
            list_text1 = []
            for pagina_num in range(num_paginas):
                pagina = lector_pdf.getPage(pagina_num)
                text = pagina.extractText()
                list_text1.append(text)
            cadena1 = " ".join(list_text1)
    except Exception as e:
        raise Exception(f"Error al leer el archivo PDF: {e}")

    # Procesando texto extraído del PDF
    patron_cupo = r'\bcertificado de cupo con número de autorización \w+/\w+'
    match_cupo = re.findall(patron_cupo, cadena1)
    match_cupo3 = match_cupo[0].split(' ')[-1] if match_cupo else 'N/A' 
    
    patron_pieza = r'\w+ Pieza.'
    match_pieza = re.findall(patron_pieza, cadena1)
    match_pieza3 = match_pieza[0][8:].replace(' Pieza.', '') if match_pieza else 'N/A'
    
# Extracción de texto del PDF 2 
    try:
        with open(PDF_sec_economia_path2, 'rb') as archivo_pdf:
            lector_pdf = PdfFileReader(archivo_pdf)
            num_paginas = lector_pdf.getNumPages()
            list_text1 = []
            for pagina_num in range(num_paginas):
                pagina = lector_pdf.getPage(pagina_num)
                text = pagina.extractText()
                list_text1.append(text)
            cadena2 = " ".join(list_text1)
    except Exception as e:
        raise Exception(f"Error al leer el archivo PDF: {e}")

    # Procesando texto extraído del PDF
    patron_cupo2 = r'\bcertificado de cupo con número de autorización \w+/\w+'
    match_cupo2 = re.findall(patron_cupo2, cadena2)
    match_cupo32 = match_cupo2[0].split(' ')[-1] if match_cupo2 else 'N/A' 
    
    patron_pieza2 = r'\w+ Pieza.'
    match_pieza2 = re.findall(patron_pieza2, cadena2)
    match_pieza32 = match_pieza2[0][8:].replace(' Pieza.', '') if match_pieza2 else 'N/A'
    
        
    
    
    # Prepare for summaries
    summary_data = {
        "Audi": 0,
        "Bentley": 0,
        "Porsche": 0,
        "Seat": 0,
        "Volkswagen": 0,
        "India": 0,
        "Sudafrica": 0,
    }

    # Generación de archivos mensuales
    with pd.ExcelWriter(output_file) as writer:
        previous_sheet_name = None

        for month in range(1, 13):
            month_str = f"{month:02d}"
            print(month_str)
            # Shift month by -1 (so Feb is treated as Jan, etc.)
            shifted_month_str = f"{(month) % 12 + 1:02d}"
            
            # Filtrar por mes en FECHA PEDIMENTO
            df_pedimentos_mes = df_pedimentos[df_pedimentos['FECHA PEDIMENTO'].str[4:6] == shifted_month_str]
            
            # Filtrando datos por país y marca
            df_pedimentos_India = df_pedimentos_mes[df_pedimentos_mes['PAIS'] == 'IND']
            df_pedimentos_Sudafrica = df_pedimentos_mes[df_pedimentos_mes['PAIS'] == 'ZAF']
            df_pedimentos_NO_India = df_pedimentos_mes[df_pedimentos_mes['PAIS'] != 'IND']
            df_pedimentos_NO_Sudafrica = df_pedimentos_NO_India[df_pedimentos_NO_India['PAIS'] != 'ZAF']
            df_pedimentos_N = df_pedimentos_NO_Sudafrica[df_pedimentos_NO_Sudafrica['J y N'] == 'N']
            df_pedimentos_N_excepcion = df_pedimentos_N[df_pedimentos_N['FRACCION'] != 8703800100]
            
            df_pedimento_Audi = df_pedimentos_N_excepcion[df_pedimentos_N_excepcion['MARCA'] == 'AUDI']
            df_pedimento_Bentley = df_pedimentos_N_excepcion[df_pedimentos_N_excepcion['MARCA'] == 'BENTLEY']
            df_pedimento_Porsche = df_pedimentos_N_excepcion[df_pedimentos_N_excepcion['MARCA'] == 'PORSCHE']
            df_pedimento_Seat = df_pedimentos_N_excepcion[df_pedimentos_N_excepcion['MARCA'] == 'SEAT']
            df_pedimento_Volkswagen = df_pedimentos_N_excepcion[df_pedimentos_N_excepcion['MARCA'] == 'VOLKSWAGEN']
            
            
            #Calculo para totales anuales
            summary_data["Audi"] += len(df_pedimento_Audi)
            summary_data["Bentley"] += len(df_pedimento_Bentley)
            summary_data["Porsche"] += len(df_pedimento_Porsche)
            summary_data["Seat"] += len(df_pedimento_Seat)
            summary_data["Volkswagen"] += len(df_pedimento_Volkswagen)
            summary_data["India"] += len(df_pedimentos_India)
            summary_data["Sudafrica"] += len(df_pedimentos_Sudafrica)


            # Cálculos
            suma_descargo = (len(df_pedimentos_India) + len(df_pedimentos_Sudafrica) + 
                             len(df_pedimento_Bentley) + len(df_pedimento_Audi) + 
                             len(df_pedimento_Porsche) + len(df_pedimento_Seat) + 
                             len(df_pedimento_Volkswagen))
            
            Subtotal_desglose_Europa = (len(df_pedimento_Bentley) + len(df_pedimento_Audi) + 
                                        len(df_pedimento_Porsche) + len(df_pedimento_Seat) + 
                                        len(df_pedimento_Volkswagen))
            Diferencia_cupo_desgloses = float(match_pieza3) - suma_descargo
            
            # Formato del estadístico para el mes
            list_col_A = ['', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', 
                          'TOTAL CUPO ' + match_pieza3, match_cupo3]
            list_col_B = ['', f'Periodo {month_str} 2024', '', f'Descargo {month_str} 2024', '', 'DESGLOSE EUROPA:', 
                          'Audi', 'Bentley', 'Porsche', 'Seat', 'Vehiculos Comerciales', 'Subtotal',
                          'Desglose INDIA:', 'India', 'Subtotal', 'Desglose SUDAFRICA', 'Sudafrica', 'Subtotal',
                          'TOTAL', '', 'CUPO', 'SALDO']
            list_col_C = ['', '',  '', suma_descargo, '', '', len(df_pedimento_Audi), len(df_pedimento_Bentley), 
                          len(df_pedimento_Porsche), len(df_pedimento_Seat), len(df_pedimento_Volkswagen),
                          Subtotal_desglose_Europa, '', len(df_pedimentos_India), len(df_pedimentos_India), 
                          '', len(df_pedimentos_Sudafrica), len(df_pedimentos_Sudafrica), int(suma_descargo), '', 
                          int(match_pieza3), int(Diferencia_cupo_desgloses)]
            list_col_D = ['', '', '', 'vehiculos', '', '', '', '', '', '', '', 'vehiculos', '', '', '', '', '', '',
                          'vehiculos', '', 'vehiculos', '']

            # DataFrame for the monthly sheet
            df_col_completo_pestana = pd.DataFrame([list_col_A, list_col_B, list_col_C, list_col_D]).T

            # Apply formula in C22
            if month == 1:
                df_col_completo_pestana.iloc[21, 2] = '=C22-C20'
            else:
                df_col_completo_pestana.iloc[20, 2] = f"='{previous_sheet_name}'!C23"
                df_col_completo_pestana.iloc[21, 2] = '=C22-C20'

            
            # Save in the sheet for each month
            sheet_name = f'Mes_{month_str}'
            df_col_completo_pestana.to_excel(writer, sheet_name=sheet_name, index=False)
            previous_sheet_name = sheet_name  # Update to current sheet name for next iteration
        # Summary sheet
            TotalCoches = summary_data["Audi"]+ summary_data["Bentley"]+ summary_data["Porsche"]+ summary_data["Seat"]+ summary_data["Volkswagen"]+ summary_data["India"]+ summary_data["Sudafrica"]
            df_summary = pd.DataFrame({
            "Categoria": ["Audi", "Bentley", "Porsche", "Seat", "Volkswagen", "India", "Sudafrica",
                          'SUMA VEHÍCULOS TODAS LAS MARCAS SIN ORIGEN =','SALDO AGOSTO CUPO PRODUCCIÓN 2024',
                         '+','SALDO CUPO INVERSIÓN 2024',"'=",'SALDO TOTAL CUPOS UNILATERAL'],
            "Total": [summary_data["Audi"], summary_data["Bentley"], summary_data["Porsche"], 
                      summary_data["Seat"], summary_data["Volkswagen"], summary_data["India"], 
                      summary_data["Sudafrica"],
                      TotalCoches ,int(match_pieza32)-TotalCoches,'',int(match_pieza3),'',int(match_pieza32)-TotalCoches+int(match_pieza3)                     ]
            })
            
            
            #Asignacion para aplicados...
            SumaTotalCupos=int(match_pieza3)+ int(match_pieza32)
            if(int(match_pieza32)-TotalCoches <=0):
                AplicadoProduccion= int(match_pieza32)
                SaldoProduccion=0
                
                AplicadoInversion=TotalCoches-int(match_pieza32)
                SaldoInversion=int(match_pieza3)-AplicadoInversion
                
                MontoAplicado=AplicadoProduccion+AplicadoInversion
                PorcentajeAplicado=MontoAplicado*100/SumaTotalCupos
            else:
                AplicadoProduccion= TotalCoches
                SaldoProduccion= int(match_pieza32)-TotalCoches
                
                AplicadoInversion=0
                SaldoInversion=int(match_pieza3)
                MontoAplicado=AplicadoProduccion+AplicadoInversion
                PorcentajeAplicado=MontoAplicado*100/SumaTotalCupos
            
                
                

            sumarySec = [
                ['', 'RESUMEN CUPOS AÑO 2024', ''],
                ['PRODUCCION', 'INVERSION', 'SUMA TOTAL CUPOS'],
                [int(match_pieza32), int(match_pieza3), SumaTotalCupos],
                ['APLICADO', 'APLICADO', 'MONTO APLICADO'],
                [AplicadoProduccion, AplicadoInversion, MontoAplicado],
                ['SALDO', 'SALDO', 'PORCENTAJE APLICADO'],
                [SaldoProduccion, SaldoInversion, f"{PorcentajeAplicado:.2f}%"]
            ]
            df_sumarySec = pd.DataFrame(sumarySec)
            df_summary.to_excel(writer, sheet_name='Resumen', index=False)

            df_sumarySec.to_excel(writer, sheet_name='Resumen', index=False, startrow=0, startcol=len(df_summary.columns) + 2)

            
    print(f"Estadístico mensual guardado en '{output_file}' para cada mes del año.")

def Concentrado2(file_path_concentrado1, file_path_PDFs):
    # 1er concentrado
    df_concentrado1 = pd.read_excel(file_path_concentrado1, engine='openpyxl', dtype=str)

    # Informacion de los PDFs extraidos
    df_pdfs = pd.read_excel(file_path_PDFs, engine='openpyxl', dtype=str)
    
    # Columnas por rectificar del Concentrado
    list_fact_concentrado = list(df_concentrado1['FACT'])
    list_chasis_concentrado = list(df_concentrado1['CHASIS'])
    list_precio_concentrado = list(df_concentrado1['PRECIO'])
    list_pais_concentrado = list(df_concentrado1['PAIS'])
    list_index_concentrado = list(df_concentrado1.index)
    
    # Columnas que se usaran del PDFs
    list_fact_pdfs = list(df_pdfs['Factura'])
    list_chasis_pdfs = list(df_pdfs['Chasis'])
    list_precio_pdfs = list(df_pdfs['Amount'])
    list_pais_pdfs = list(df_pdfs['Pais'])
    list_JyN_pdfs = list(df_pdfs['J y N'])
    
    list_index_encontrado = []
    list_JyN_revision = []
    list_pais_revision = []
    list_precio_revision = []
    list_fact_revision = []
        
    for i in range(len(list_fact_concentrado)):
        for j in range(len(list_fact_pdfs)):
            # Convert to string and slice safely, ignoring NaN values
            fact_pdf_str = str(list_fact_pdfs[j])[:8] if pd.notna(list_fact_pdfs[j]) else ''
            if str(list_fact_concentrado[i]) == fact_pdf_str and list_chasis_concentrado[i] == list_chasis_pdfs[j]:
                print('match: ', fact_pdf_str)
                list_index_encontrado.append(i)
                list_pais_revision.append(list_pais_pdfs[j])
                list_precio_revision.append(list_precio_pdfs[j])
                list_fact_revision.append(list_fact_pdfs[j])
                print('pais: ', list_JyN_pdfs[j])
                # Condiciones especiales
                if list_JyN_pdfs[j] == 'C.O' and list_pais_pdfs[j] == 'USA':
                    list_JyN_revision.append('J')
                elif list_JyN_pdfs[j] == 'C.O' and list_pais_pdfs[j] == 'BRA':
                    list_JyN_revision.append('N')
                elif list_JyN_pdfs[j] == 'CUPO' and list_pais_pdfs[j] == 'IND': 
                    list_JyN_revision.append('N')
                else:
                    list_JyN_revision.append(list_JyN_pdfs[j])

                    
    # Si no encuentra coincidencia entre ambos archivos, este proceso hara que se llene con un nan, respetando la posicion del index que le corresponda
    list_JyN_revision_2 = []
    list_pais_revision_2 = []
    list_precio_revision_2 = []
    list_fact_revision_2 = []
    list_pais_witness = []

    for i in range(len(list_index_concentrado)):
        matched = False
        for j in range(len(list_index_encontrado)):
            if list_index_concentrado[i] == list_index_encontrado[j]:
                list_JyN_revision_2.append(list_JyN_revision[j])
                list_pais_revision_2.append(list_pais_revision[j])
                list_precio_revision_2.append(list_precio_revision[j])
                list_fact_revision_2.append(list_fact_revision[j])
                matched = True
                # Witness contains 'sin cambio' if there's no change in 'PAIS', otherwise contains new 'PAIS'
                if list_pais_revision_2[-1] == list_pais_concentrado[i]:
                    list_pais_witness.append('sin cambio')
                else:
                    list_pais_witness.append(list_pais_revision_2[-1])
                
        if not matched:
            list_JyN_revision_2.append(np.nan)
            list_pais_revision_2.append(list_pais_concentrado[i])
            list_precio_revision_2.append(list_precio_concentrado[i])
            list_fact_revision_2.append(list_fact_concentrado[i])
            list_pais_witness.append('sin cambio')

    df_concentrado1['J y N'] = list_JyN_revision_2
    df_concentrado1['PAIS'] = list_pais_revision_2
    df_concentrado1['PRECIO'] = list_precio_revision_2
    df_concentrado1['FACT'] = list_fact_revision_2
    df_concentrado1['PAIS_WITNESS'] = list_pais_witness  # Add witness column

    return [df_concentrado1, df_pdfs]


In [28]:
#dwh = '/home/gestell3/Desktop/TODAS FACTURAS VW/P1 - PDFs/DWH General/PedimentoGeneral2024'
dwh='/home/gestell3/Desktop/TODAS FACTURAS VW/P1 - PDFs/DWH General/Pedimento_Proyecto_ENE24.txt'
divisiones = '/home/gestell3/Desktop/TODAS FACTURAS VW/divisonesIncrementales_FormatoNuevo.xlsx'
facturas = '/home/gestell3/Desktop/ExtraccionGralInc23.xlsx'
output1 = '/home/gestell3/Desktop/PRUEBAS/pruebaConcentrado1.xlsx'
output2 = '/home/gestell3/Desktop/PRUEBAS/pruebaConcentrado2.xlsx'
pdfInversion = '/home/gestell3/Desktop/TODAS FACTURAS VW/P1 - PDFs/DWH General/Certificado de cupo de importación 2024 - 23VEH001245-2134_Producción.pdf'

pdfProduccion = '/home/gestell3/Desktop/TODAS FACTURAS VW/P1 - PDFs/DWH General/Certificado de cupo de importación - 24VEH000491-2134_Inversión.pdf'

output3 = '/home/gestell3/Desktop/PRUEBAS/pruebaEstadistico.xlsx'

concentrado = Concentrado1Mod(dwh, divisiones)
concentrado[0].to_excel(output1, sheet_name='final')
concentrado[0]
conc2=Concentrado2(output1,facturas)
conc2[0].to_excel(output2, sheet_name="concentrado2", index=False)
estadistico_v2(output2,pdfProduccion,output3, pdfInversion)
#conc2[0]


01
02
03
04
05
06
07
08
09
10
11
12
Estadístico mensual guardado en '/home/gestell3/Desktop/PRUEBAS/pruebaEstadistico.xlsx' para cada mes del año.


In [17]:
import pandas as pd

# Load the Excel files, specifying only the first sheet for the file with multiple sheets
df1 = pd.read_excel('/home/gestell3/Desktop/PRUEBAS/Ahorro 2024.xlsx', sheet_name=0)  # Load the first sheet (Master file)
df2 = pd.read_excel('/home/gestell3/Desktop/PRUEBAS/pruebaConcentrado2.xlsx')  # File to compare
df3 = pd.read_excel('/home/gestell3/Desktop/ExtraccionGeneralTODO.xlsx')  # Another file to compare

# Define the key column
key_column = 'Pedimento'
key_column2 = 'FACTSEAT'


print("Columns in df1:", df1.columns)
print("Columns in df2:", df2.columns)
print("Columns in df3:", df3.columns)

# Find records in df1 missing in df2 and df3
missing_in_df2 = df1[~df1[key_column].isin(df2[key_column])]
missing_in_df3 = df1[~df1['FACTSEAT'].isin(df3['Factura'])]

# Get unique Pedimento values for these missing records
unique_pedimento_missing_in_df2 = missing_in_df2[key_column].dropna().unique()

# Combine unique Pedimento values into a DataFrame
unique_pedimento_missing = pd.DataFrame({
    "Missing in concentrado2": pd.Series(unique_pedimento_missing_in_df2)
})

#get unique facturias
unique_fact_missing_in_df=missing_in_df3['FACTSEAT'].dropna().unique()
unique_fact_missing= pd.DataFrame({
    "Missing in extraccion facturas":pd.Series(unique_fact_missing_in_df)
})

# Save results to a new Excel file
with pd.ExcelWriter('/home/gestell3/Desktop/PRUEBAS/missing_from_master_results.xlsx') as writer:
    unique_pedimento_missing.to_excel(writer, sheet_name='Missing Pedimento', index=False)
    unique_fact_missing.to_excel(writer, sheet_name='Missing Factura', index=False)

print('Process complete')


Columns in df1: Index(['ID', 'AUTO', 'FACTSEAT', 'FECFAC', 'CHASIS', 'J y N', 'PRECIO', 'TIPO',
       'FRACCION', 'PAIS', 'Patente', 'Pedimento', 'Fecha pedimento', 'FLETES',
       'SEGUROS', 'ADUANA', 'F y S (DLLS)', 'PRECIO + F y S', 'IGI', 'IVA',
       'DTA', 'TOTAL DLLS', 'MENOS DTA', 'IGI.1', 'DTA.1', 'TOTAL DLLS.1'],
      dtype='object')
Columns in df2: Index(['Unnamed: 0', 'ID', 'AUTO', 'FACT', 'FECFACT', 'CHASIS', 'PRECIO',
       'TIPO', 'FRACCION', 'PAIS', 'PATENTE', 'Pedimento', 'FECHA PEDIMENTO',
       'FLETES', 'SEGUROS', 'ADUANA', 'MARCA', 'J y N', 'PAIS_WITNESS'],
      dtype='object')
Columns in df3: Index(['Factura', 'Pais', 'Fecha', 'Auto', 'CHASIS', 'J y N', 'Amount',
       'Moneda', 'Leyenda'],
      dtype='object')
Process complete
